# 📊 Statistical Models

**✍️ Author:** Hayriye Anıl  
**📘 Blog Series:** Time Series Analysis & Forecasting  

## 🎯 Purpose

This notebook implements and evaluates classical statistical forecasting models for temperature prediction. These models leverage time series properties like autocorrelation, trends, and seasonality to generate forecasts that should outperform simple baseline approaches.

## 📂 Contents

1. **Data Preparation**
   - Load processed weather dataset
   - Split data into training and test sets
   - Check ACF and PACF plots

2. **Run Auto ARIMA model once**
   - Determine optimal ARIMA parameters automatically using Auto ARIMA

3. **Statistical Model Implementation**
   - **ARIMA**: AutoRegressive Integrated Moving Average for univariate forecasting
   - **SARIMA**: Seasonal ARIMA to capture seasonal patterns in the data
   - **SARIMAX**: Seasonal ARIMA with eXogenous variables for enhanced predictions
   - **VAR**: Vector AutoRegression for multivariate time series forecasting

4. **Backtesting Framework**
   - Implement sliding window and extending window techniques
   - Generate multi-horizon forecasts (H+24, H+48, H+72, H+96, H+120)
   - Evaluate model performance across different forecast horizons

5. **Model Evaluation & Comparison**
   - Calculate performance metrics (MAE, RMSE, Bias, NMAE, NRMSE)
   - Compare statistical models against baseline benchmarks
   - Analyze forecast accuracy degradation over time


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

import pandas as pd
import numpy as np
from datetime import timedelta

from data.paths import PROCESSED_DATA_DIR, STATISTICAL_MODEL_DIR, BASELINE_MODEL_DIR, PERFORMANCE_DIR
from src import ts_analysis as ts 
from src import statistical_models as sm
from src import backtesting as bt
from src import plots as plot
from src import kpis as kpi

### Custom Functions

In [ ]:
def run_model_with_bactesting(train_set: pd.DataFrame,
                              test_set: pd.DataFrame,
                              target: str, 
                              model_func, 
                              technique: str, 
                              forecast_days: int, 
                              timestamps_per_day: int, 
                              forecast_horizon: int,
                              exog_cols=list | None,
                              **model_params,
                                    ) -> pd.DataFrame:
    """Run ARIMA model with backtesting for each day in the test set and collect results.
    
    Args:
        train_set (pd.DataFrame): The training dataset.
        test_set (pd.DataFrame): The test dataset.
        target (str): The target variable to forecast.
        model_func (callable): The model function to run.
        technique (str): The backtesting technique to apply ('sliding_window' or 'extending_window').
        forecast_days (int): The number of days to forecast.
        timestamps_per_day (int): The number of timestamps in a day (e.g., 24 for hourly data).
        forecast_horizon (int): The total number of timestamps to forecast
        **model_params: Additional parameters to pass to the model function.
    
    Returns:
        pred_result_frame (pd.DataFrame): DataFrame containing forecast results for each day.
    """
    result_frames = []
    test_set_unique_dates = np.unique(test_set.index.date)
    for date in test_set_unique_dates:
        print("TRAIN DATE:", date)
        forecast_timesteps = pd.date_range(start=date, end=date + pd.DateOffset(days=forecast_days), freq='h', inclusive='left')
        print(forecast_timesteps[0], forecast_timesteps[-1])
        forecast_test_data = test_set[test_set.index.isin(forecast_timesteps)][[target]]
        # Run forecasting model for each day
        if forecast_test_data.shape == (forecast_horizon, 1):
            # Handle exogenous variables for SARIMAX
            if exog_cols:
                model_params['exog_train'] = train_set[exog_cols]
                model_params['exog_future'] = test_set[test_set.index.isin(forecast_timesteps)][exog_cols]
            
            pred_frame = model_func(train_set, forecast_test_data, target, **model_params) 
                                                                       # range(24, 121, 24)], 24)
            pred_frame["horizon"] = np.repeat([f'H+{k}' for k in range(timestamps_per_day, forecast_horizon+1, timestamps_per_day)], timestamps_per_day)
            pred_frame['actual'] = test_set.loc[test_set.index.isin(pred_frame.index), target].values
            pred_frame['forecast_run_date'] = date - timedelta(days=1)
            result_frames.append(pred_frame)
            train_set = bt.backtesting(train_set, test_set, date, technique)
            print("TRAIN start and end date:", np.unique(train_set.index.date)[0], np.unique(train_set.index.date)[-1])
            print("TRAIN SHAPE:", train_set.shape)

    return pd.concat(result_frames)

In [ ]:
def run_var_model_with_backtesting(train_set: pd.DataFrame,
                                   test_set: pd.DataFrame,
                                   variables: list,
                                   technique: str,
                                   forecast_days: int,
                                   timestamps_per_day: int,
                                   forecast_horizon: int,
                                   p: int) -> pd.DataFrame:
    """
    Run VAR model with backtesting. VAR is multivariate forecasting method,
    it models multiple time series together, letting each variable use the past
    of all variables.

    Args:
        train_set (pd.DataFrame): Training data.
        test_set (pd.DataFrame): Test data.
        variables (list): List of variable names to forecast.
        technique (str): Backtesting technique ('sliding_window' or 'extending_window').
        forecast_days (int): Number of days to forecast.
        timestamps_per_day (int): Number of timestamps per day.
        forecast_horizon (int): Total forecast horizon in hours.
        p (int): Number of lags for VAR model.

    Returns:
        pd.DataFrame: Concatenated predictions for all dates.
    """
    result_frames = []
    test_set_unique_dates = np.unique(test_set.index.date)
    for date in test_set_unique_dates:
        print("TRAIN DATE:", date)
        forecast_timesteps = pd.date_range(start=date, end=date + pd.DateOffset(days=forecast_days), freq='h', inclusive='left')
        print(forecast_timesteps[0], forecast_timesteps[-1])
        forecast_test_data = test_set[test_set.index.isin(forecast_timesteps)][variables]
        # Run forecasting model for each day
        if forecast_test_data.shape == (forecast_horizon, len(variables)):
            
            pred_frame = sm.run_var_model(train_set, forecast_test_data, variables, p, forecast_timesteps) 
                                                                        # range(24, 121, 24)], 24)
            pred_frame["horizon"] = np.repeat([f'H+{k}' for k in range(timestamps_per_day, forecast_horizon+1, timestamps_per_day)], timestamps_per_day)
            # add actual values for each variable in its own column
            for var in variables:
                pred_frame[f'actual_{var}'] = test_set.loc[test_set.index.isin(pred_frame.index), var].values
            pred_frame['forecast_run_date'] = date - timedelta(days=1)
            result_frames.append(pred_frame)
            train_set = bt.backtesting(train_set, test_set, date, technique)
            print("TRAIN start and end date:", np.unique(train_set.index.date)[0], np.unique(train_set.index.date)[-1])
            print("TRAIN SHAPE:", train_set.shape)

    return pd.concat(result_frames)

In [ ]:
def model_performance_over_horizon(predictions_frame: pd.DataFrame, actual_col: str, pred_col: str) -> pd.DataFrame:
    """Calculate model performance metrics for each forecast horizon.
    
    Args:
        predictions_frame (pd.DataFrame): DataFrame containing actual and predicted values along with forecast horizons.
        actual_col (str): The name of the column containing actual values.
        pred_col (str): The name of the column containing predicted values.
        
    Returns:
        performances_frame (pd.DataFrame): DataFrame containing performance metrics for each forecast horizon.
    """
    horizon_performances = []
    horizons = predictions_frame['horizon'].unique()
    for horizon in horizons:
        horizon_data = predictions_frame[predictions_frame['horizon'] == horizon]
        metric_value = kpi.metrics_summary(horizon_data, actual_col, pred_col)
        metric_value['horizon'] = horizon  
        horizon_performances.append(metric_value)

    performances_frame = pd.concat(horizon_performances, axis=0)
    return performances_frame

### Data Preparation

In [ ]:
dataset = pd.read_csv(f"{PROCESSED_DATA_DIR}/processed_dataset.csv", index_col=0, parse_dates=True)
dataset

In [ ]:
target = "temperature_2m (°C)"
data = dataset[[target]]
data = data.asfreq('h')
data

In [ ]:
train_set = data[data.index < '2025-01-01']
test_set = data[(data.index >= '2025-01-01') & (data.index <= '2026-01-01')] # 1 year test set

In [ ]:
ts.test_stationarity(dataset, target)

In [ ]:
ts.plot_autocorrelation_function(dataset, "temperature_2m (°C)", lags=24)

In [ ]:
ts.plot_partial_correlation_function(dataset, "temperature_2m (°C)", lags=24)

### Run Auto ARIMA model once

In [ ]:
forecast_days = 5
timestamps_per_day = 24 # 1 day is 24 hour
forecast_horizon = forecast_days * timestamps_per_day  # 120 time steps

In [ ]:
order, aic, pred_frame = sm.run_auto_arima_model(train_set, test_set, "temperature_2m (°C)")

### Statistical Models Implementation

#### ARIMA Model

In [38]:
arima_sliding_window_preds = run_model_with_bactesting(train_set, 
                                                       test_set, 
                                                       target, 
                                                       sm.run_arima_model, 
                                                       "sliding_window", 
                                                       forecast_days, 
                                                       timestamps_per_day, 
                                                       forecast_horizon, 
                                                       order=(2,1,4))
arima_sliding_window_preds.to_csv(f"{STATISTICAL_MODEL_DIR}/arima_sliding_window_forecast_results.csv")

NameError: name 'run_model_with_bactesting' is not defined

In [ ]:
# Define train and test sets again to reset them for the next model
train_set = data[data.index < '2025-01-01']
test_set = data[(data.index >= '2025-01-01') & (data.index <= '2026-01-01')] # 1 year test set

In [ ]:
arima_extending_window_preds = run_model_with_bactesting(train_set, 
                                                         test_set, 
                                                         target, 
                                                         sm.run_arima_model, 
                                                         "extending_window", 
                                                         forecast_days, 
                                                         timestamps_per_day, 
                                                         forecast_horizon, 
                                                         order=(2,1,4))
arima_extending_window_preds.to_csv(f"{STATISTICAL_MODEL_DIR}/arima_extending_window_forecast_results.csv")

In [ ]:
arima_sliding_window_preds = pd.read_csv(f"{STATISTICAL_MODEL_DIR}/arima_sliding_window_forecast_results.csv", 
                                         index_col=0, parse_dates=True)
arima_sliding_window_preds

In [ ]:
arima_extending_window_preds = pd.read_csv(f"{STATISTICAL_MODEL_DIR}/arima_extending_window_forecast_results.csv", 
                                         index_col=0, parse_dates=True)
arima_extending_window_preds

In [ ]:
arima_sliding_performance = model_performance_over_horizon(arima_sliding_window_preds, "actual", "prediction")
arima_sliding_performance

In [ ]:
arima_extending_performance = model_performance_over_horizon(arima_extending_window_preds, "actual", "prediction")
arima_extending_performance

In [ ]:
melted_sliding_window_performance = pd.melt(arima_sliding_performance, id_vars=['horizon'], var_name='metric', value_name='score')
plot.plot_bar_group(melted_sliding_window_performance,
                    "metric",
                    "score",
                    "horizon",
                    "score",
                    "ARIMA Metric Performance by Horizon (Sliding Window)")

In [ ]:
melted_extending_window_performance = pd.melt(arima_extending_performance, id_vars=['horizon'], var_name='metric', value_name='score')
plot.plot_bar_group(melted_extending_window_performance,
                    "metric",
                    "score",
                    "horizon",
                    "score",
                    "ARIMA Metric Performance by Horizon (Extending Window)")

In [ ]:
plot.plot_line_group(arima_extending_window_preds, 
                     arima_extending_window_preds.index, 
                    ["actual", "prediction"], 
                    "horizon",
                    "Time", 
                    "°C", 
                    "ARIMA model Actual vs Predicted by Horizon (Extending Window)")

In [ ]:
metrics = ["MAE (°C)", "RMSE (°C)", "Bias (°C)", "NMAE", "NRMSE"]
plot.compare_performance_by_horizon(arima_sliding_performance, 
                                    arima_extending_performance, 
                                    metrics,
                                    "ARIMA")

#### SARIMA model

In [ ]:
train_set = data[data.index < '2025-01-01']
test_set = data[(data.index >= '2025-01-01') & (data.index <= '2026-01-01')] # 1 year test set

In [ ]:
order = (2, 1, 4)
seasonal_order = (1, 0, 1, 24)

sarima_sliding_preds = run_model_with_bactesting(train_set, 
                                                       test_set, 
                                                       target,
                                                       sm.run_sarimax_model,
                                                       "sliding_window",
                                                       forecast_days,
                                                       timestamps_per_day,
                                                       forecast_horizon,
                                                       order=order,
                                                       seasonal_order=seasonal_order)
sarima_sliding_preds.to_csv(f"{STATISTICAL_MODEL_DIR}/sarima_sliding_window_forecast_results.csv")

In [ ]:
# Define train and test sets again to reset them for the next model
train_set = data[data.index < '2025-01-01']
test_set = data[(data.index >= '2025-01-01') & (data.index <= '2026-01-01')] # 1 year test set

In [ ]:
sarima_extending_preds = run_model_with_bactesting(train_set, 
                                                       test_set, 
                                                       target,
                                                       sm.run_sarimax_model,
                                                       "extending_window",
                                                       forecast_days,
                                                       timestamps_per_day,
                                                       forecast_horizon,
                                                       order=order,
                                                       seasonal_order=seasonal_order)
sarima_extending_preds.to_csv(f"{STATISTICAL_MODEL_DIR}/sarima_extending_window_forecast_results.csv")

In [ ]:
sarima_sliding_preds = pd.read_csv(f"{STATISTICAL_MODEL_DIR}/sarima_sliding_window_forecast_results.csv", 
                                         index_col=0, parse_dates=True)
sarima_sliding_preds

In [ ]:
sarima_extending_preds = pd.read_csv(f"{STATISTICAL_MODEL_DIR}/sarima_extending_window_forecast_results.csv", 
                                         index_col=0, parse_dates=True)
sarima_extending_preds

In [ ]:
sarima_sliding_performance = model_performance_over_horizon(sarima_sliding_preds, "actual", "prediction")
sarima_sliding_performance

In [ ]:
sarima_extending_performance = model_performance_over_horizon(sarima_extending_preds, "actual", "prediction")
sarima_extending_performance

In [ ]:
melted_sliding_performance = pd.melt(sarima_extending_performance, id_vars=['horizon'], var_name='metric', value_name='score')
plot.plot_bar_group(melted_sliding_performance,
                    "metric",
                    "score",
                    "horizon",
                    "score",
                    "SARIMA Metric Performance by Horizon (Sliding Window)")

In [ ]:
plot.plot_line_group(sarima_sliding_preds, 
                     sarima_sliding_preds.index, 
                    ["actual", "prediction"], 
                    "horizon",
                    "Time", 
                    "°C", 
                    "SARIMA model Actual vs Predicted by Horizon (Sliding Window)")

In [ ]:
melted_extending_performance = pd.melt(sarima_extending_performance, id_vars=['horizon'], var_name='metric', value_name='score')
plot.plot_bar_group(melted_extending_performance,
                    "metric",
                    "score",
                    "horizon",
                    "score",
                    "SARIMA Metric Performance by Horizon (Extending Window)")

In [ ]:
plot.plot_line_group(sarima_extending_preds, 
                     sarima_extending_preds.index, 
                    ["actual", "prediction"], 
                    "horizon",
                    "Time", 
                    "°C", 
                    "SARIMA model Actual vs Predicted by Horizon (Extending Window)")

In [ ]:
metrics = ["MAE (°C)", "RMSE (°C)", "Bias (°C)", "NMAE", "NRMSE"]
plot.compare_performance_by_horizon(sarima_sliding_performance, 
                                    sarima_extending_performance, 
                                    metrics,
                                    "SARIMA")

#### SARIMAX model

In [ ]:
target = "temperature_2m (°C)"
exog_cols = ['relative_humidity_2m (%)', 'wind_speed_10m (km/h)']
order = (2, 1, 4)
seasonal_order = (1, 0, 1, 24)
data_with_external_variables = dataset[['relative_humidity_2m (%)', 'wind_speed_10m (km/h)', target]]
data_with_external_variables = data_with_external_variables.asfreq('h')
data_with_external_variables

In [ ]:
train_set = data_with_external_variables[data_with_external_variables.index < '2025-01-01']
test_set = data_with_external_variables[(data_with_external_variables.index >= '2025-01-01') & (data_with_external_variables.index <= '2026-01-01')] # 1 year test set

In [ ]:
sarimax_sliding_preds = run_model_with_bactesting(train_set, 
                                                        test_set, 
                                                        target, 
                                                        sm.run_sarimax_model,
                                                        "sliding_window", 
                                                        forecast_days, 
                                                        timestamps_per_day,
                                                        forecast_horizon, 
                                                        exog_cols=exog_cols,
                                                        order=order, 
                                                        seasonal_order=seasonal_order)
sarimax_sliding_preds.to_csv(f"{STATISTICAL_MODEL_DIR}/sarimax_sliding_window_forecast_results.csv")

In [ ]:
# Define train and test sets again to reset them for the next model
train_set = data_with_external_variables[data_with_external_variables.index < '2025-01-01']
test_set = data_with_external_variables[(data_with_external_variables.index >= '2025-01-01') & (data_with_external_variables.index <= '2026-01-01')] # 1 year test set

In [ ]:
sarimax_extending_preds = run_model_with_bactesting(train_set, 
                                                          test_set, 
                                                          target, 
                                                          sm.run_sarimax_model,
                                                          "extending_window", 
                                                          forecast_days, 
                                                          timestamps_per_day,
                                                          forecast_horizon, 
                                                          exog_cols=exog_cols,
                                                          order=order, 
                                                          seasonal_order=seasonal_order)
sarimax_extending_preds.to_csv(f"{STATISTICAL_MODEL_DIR}/sarimax_extending_window_forecast_results.csv")

In [ ]:
sarimax_sliding_preds = pd.read_csv(
f"{STATISTICAL_MODEL_DIR}/sarimax_sliding_window_forecast_results.csv",
                                         index_col=0, parse_dates=True)

sarimax_sliding_preds

In [ ]:
sarimax_extending_preds = pd.read_csv(
f"{STATISTICAL_MODEL_DIR}/sarimax_extending_window_forecast_results.csv",
                                         index_col=0, parse_dates=True)

sarimax_extending_preds

In [ ]:
sarimax_sliding_performance = model_performance_over_horizon(sarimax_sliding_preds, "actual", "prediction")
sarimax_sliding_performance

In [ ]:
sarimax_extending_performance = model_performance_over_horizon(sarimax_extending_preds, "actual", "prediction")
sarimax_extending_performance

In [ ]:
melted_sliding_performance = pd.melt(sarimax_sliding_performance, id_vars=['horizon'], var_name='metric', value_name='score')
plot.plot_bar_group(melted_sliding_performance,
                    "metric",
                    "score",
                    "horizon",
                    "score",
                    "SARIMAX Metric Performance by Horizon (Sliding Window)")

In [ ]:
plot.plot_line_group(sarimax_sliding_preds, 
                     sarimax_sliding_preds.index, 
                    ["actual", "prediction"], 
                    "horizon",
                    "Time", 
                    "°C", 
                    "SARIMAX model Actual vs Predicted by Horizon (Sliding Window)")

In [ ]:
melted_extending_performance = pd.melt(sarimax_extending_performance, id_vars=['horizon'], var_name='metric', value_name='score')
plot.plot_bar_group(melted_extending_performance,
                    "metric",
                    "score",
                    "horizon",
                    "score",
                    "SARIMAX Metric Performance by Horizon (Extending Window)")

In [ ]:
plot.plot_line_group(sarimax_extending_preds, 
                     sarimax_extending_preds.index, 
                    ["actual", "prediction"], 
                    "horizon",
                    "Time", 
                    "°C", 
                    "SARIMAX model Actual vs Predicted by Horizon (Extending Window)")

In [ ]:
metrics = ["MAE (°C)", "RMSE (°C)", "Bias (°C)", "NMAE", "NRMSE"]
plot.compare_performance_by_horizon(sarimax_sliding_performance, 
                                    sarimax_extending_performance, 
                                    metrics,
                                    "SARIMAX")

#### VAR

In [ ]:
variables = [
    "temperature_2m (°C)",
    "relative_humidity_2m (%)",
    "wind_speed_10m (km/h)"
]
p = 24

In [ ]:
dataset_with_variables = dataset[variables]
dataset_with_variables

In [ ]:
train_set = dataset_with_variables[dataset_with_variables.index < '2025-01-01']
test_set = dataset_with_variables[(dataset_with_variables.index >= '2025-01-01') & (dataset_with_variables.index <= '2026-01-01')] # 1 year test set

In [ ]:
var_sliding_preds = run_var_model_with_backtesting(train_set, 
                                                   test_set, 
                                                   variables, 
                                                   "sliding_window", 
                                                   forecast_days, 
                                                   timestamps_per_day, 
                                                   forecast_horizon,
                                                   p)
var_sliding_preds.to_csv(f"{STATISTICAL_MODEL_DIR}/var_sliding_window_forecast_results.csv")

In [ ]:
# Define train and test sets again to reset them for the next model
train_set = dataset_with_variables[dataset_with_variables.index < '2025-01-01']
test_set = dataset_with_variables[(dataset_with_variables.index >= '2025-01-01') & (dataset_with_variables.index <= '2026-01-01')] # 1 year test set

In [ ]:
var_extending_preds = run_var_model_with_backtesting(train_set, 
                                                test_set, 
                                                variables, 
                                                "extending_window", 
                                                forecast_days, 
                                                timestamps_per_day, 
                                                forecast_horizon,
                                                p)
var_extending_preds.to_csv(f"{STATISTICAL_MODEL_DIR}/var_extending_window_forecast_results.csv")

In [ ]:
var_sliding_preds = pd.read_csv(
f"{STATISTICAL_MODEL_DIR}/var_sliding_window_forecast_results.csv",
                                    index_col=0, parse_dates=True)
var_sliding_preds

In [ ]:
var_extending_preds = pd.read_csv(
f"{STATISTICAL_MODEL_DIR}/var_extending_window_forecast_results.csv",
                                    index_col=0, parse_dates=True)
var_extending_preds

In [ ]:
metrics = ["MAE (°C)", "RMSE (°C)", "Bias (°C)", "NMAE", "NRMSE"]
for variable in variables:
    print(f"---------- Variable: {variable }-------------")
    actual = f"actual_{variable}"
    prediction = f"pred_{variable}"
    print("-------- Sliding Window --------")
    var_sliding_performance = model_performance_over_horizon(var_sliding_preds,
                                                             actual,
                                                             prediction)
    
    melted_sliding_performance = pd.melt(var_sliding_performance, id_vars=['horizon'], var_name='metric', value_name='score')
    plot.plot_bar_group(melted_sliding_performance,
                    "metric",
                    "score",
                    "horizon",
                    "score",
                    f"VAR {variable} Metric Performance by Horizon (Sliding Window)")
    
    
    print("-------- Extending Window --------")
    var_extending_performance = model_performance_over_horizon(var_extending_preds,
                                                               actual,
                                                                prediction)
   
    melted_extending_performance = pd.melt(var_extending_performance, id_vars=['horizon'], var_name='metric', value_name='score')
    plot.plot_bar_group(melted_extending_performance,
                    "metric",
                    "score",
                    "horizon",
                    "score",
                    f"VAR {variable} Metric Performance by Horizon (Extending Window)")
    
    plot.compare_performance_by_horizon(var_sliding_performance, 
                                        var_extending_performance, 
                                        metrics,
                                        "VAR")

In [ ]:
var_sliding_performance = model_performance_over_horizon(var_sliding_preds, "actual_temperature_2m (°C)", "pred_temperature_2m (°C)")
var_extending_performance = model_performance_over_horizon(var_extending_preds, "actual_temperature_2m (°C)", "pred_temperature_2m (°C)")

#### Save Model Results

In [ ]:
arima_sliding_performance["model"] = "ARIMA (Sliding Window)"
sarima_sliding_performance["model"] = "SARIMA (Sliding Window)"
sarimax_sliding_performance["model"] = "SARIMAX (Sliding Window)"
var_sliding_performance["model"] = "VAR (Sliding Window)"

In [ ]:
sliding_statistical_models_metrics = pd.concat([arima_sliding_performance, 
                                                sarima_sliding_performance, 
                                                sarimax_sliding_performance,
                                                var_sliding_performance], axis=0)
sliding_statistical_models_metrics
                                

In [ ]:
sliding_statistical_models_metrics.to_csv(f"{PERFORMANCE_DIR}/statistical_models_sliding_window_multihorizon_metrics.csv", index=False)

In [ ]:
arima_extending_performance["model"] = "ARIMA (Extending Window)"
sarima_extending_performance["model"] = "SARIMA (Extending Window)"
sarimax_extending_performance["model"] = "SARIMAX (Extending Window)"
var_extending_performance["model"] = "VAR (Extending Window)"

In [ ]:
extending_statistical_models_metrics = pd.concat([arima_extending_performance, 
                                                  sarima_extending_performance, 
                                                  sarimax_extending_performance,
                                                  var_extending_performance], axis=0)
extending_statistical_models_metrics                       

In [ ]:
extending_statistical_models_metrics.to_csv(f"{PERFORMANCE_DIR}/statistical_models_extending_window_multihorizon_metrics.csv", index=False)

### Model Evaluation & Comparison

In [ ]:
metrics = ["MAE (°C)", "RMSE (°C)", "Bias (°C)", "NMAE", "NRMSE"]

In [ ]:
plot.compare_models_by_horizon(
    {
        'ARIMA': arima_sliding_performance,
        'SARIMA': sarima_sliding_performance,
        'SARIMAX': sarimax_sliding_performance,
        'VAR': var_sliding_performance
    },
    metrics,
    'Sliding Window'
)

In [ ]:
plot.compare_models_by_horizon(
    {
        'ARIMA': arima_extending_performance,
        'SARIMA': sarima_extending_performance,
        'SARIMAX': sarimax_extending_performance,
        'VAR':  var_extending_performance
    },
    metrics,
    'Extending Window'
)

In [ ]:
plot.compare_techniques_by_model({
    'ARIMA': {
        'Sliding Window': arima_sliding_performance,
        'Extending Window': arima_extending_performance
    },
    'SARIMA': {
        'Sliding Window': sarima_sliding_performance,
        'Extending Window': sarima_extending_performance
    },
    'SARIMAX': {
        'Sliding Window': sarimax_sliding_performance,
        'Extending Window': sarimax_extending_performance
    },
    'VAR': {
        'Sliding Window': model_performance_over_horizon(var_sliding_preds, f"actual_{target}", f"pred_{target}"),
        'Extending Window': model_performance_over_horizon(var_extending_preds, f"actual_{target}", f"pred_{target}")
    },
}, metric='MAE (°C)')


In [ ]:
baseline_model_metrics = pd.read_csv(f"{PERFORMANCE_DIR}/baseline_models_multihorizon_metrics.csv")
naive_24h_model_performance = baseline_model_metrics[baseline_model_metrics['model'] == 'naive_24h']
naive_24h_model_performance

In [ ]:
comparison_metrics_between_sliding_models = pd.concat([arima_sliding_performance, 
           sarima_sliding_performance, 
           sarimax_sliding_performance,
           var_sliding_performance,
           naive_24h_model_performance], axis=0)
comparison_metrics_between_sliding_models

In [ ]:
comparison_metrics_between_extending_models = pd.concat([arima_extending_performance, 
           sarima_extending_performance, 
           sarimax_extending_performance,
           var_extending_performance,
           naive_24h_model_performance], axis=0)
comparison_metrics_between_extending_models

In [ ]:
for column in comparison_metrics_between_sliding_models.columns[:-2]:
    plot.plot_bar_group(comparison_metrics_between_sliding_models,
                        "model",
                        column,
                        "horizon",
                        column,
                        f"{column} Scores by Model and Horizon (Sliding Window)")

In [ ]:
for column in comparison_metrics_between_extending_models.columns[:-2]:
    plot.plot_bar_group(comparison_metrics_between_extending_models,
                        "model",
                        column,
                        "horizon",
                        column,
                        f"{column} Scores by Model and Horizon (Extending Window)")